# Phase 4 of 4: Evaluation Against the LoRA's Held-Out Sets

**Run order:** 4 of 4, after a Phase 3 run exists. This is the run-order phase; the roadmap in `docs/PLAN.md` §7 numbers generated-data hardening as its Phase 4.
**Produces:** `eval_wildguard_test.json` and `eval_wildguard_test_rows.jsonl` inside the Phase 3 run directory.
**Expect:** under a minute; the printed block mirrors the LoRA eval notebook's output so the two can be read side by side.

Scores a saved Phase 3 run on `lora_holdout.parquet`: WildGuardMix `wildguardtest` labelled exactly
as `safety/notebooks/prompt_injection_lora_qwen38_27b_eval.ipynb` labels it, plus the Nemotron
test-split English `jailbreaking` rows. The datagen notebook removed every exact and near-duplicate
of these rows from train, val and test, and this notebook re-checks the fingerprint of the data
build the run was trained on.

Each row is turned into the contract string (`SAFE` or `INJECTION: <reason>`), and the results are
written as `eval_wildguard_test.json` with the **same field names the LoRA eval notebook writes**
(`rows`, `accuracy`, `invalid`, `false_positives`, `safe_rows`, `false_negatives`,
`injection_rows`, `sources`), plus the probability-based metrics the LoRA cannot produce. The
per-row file `eval_wildguard_test_rows.jsonl` sits beside it, in the run directory.

No LoRA `eval_*.json` exists yet (`safety/README.md`); a head-to-head needs that run. Until then this
file is one side of the comparison.


## 1. Configuration

`RUN_ID = None` picks the newest Phase 3 run directory under `models/`.


In [1]:
import json
import os
import platform
import re
import sys
from collections import Counter
from contextlib import nullcontext
from datetime import datetime, timezone
from importlib.metadata import version as package_version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    average_precision_score, brier_score_loss, confusion_matrix, f1_score, log_loss,
    precision_recall_curve, precision_score, recall_score, roc_auc_score,
)
from torch import nn
from torch.nn.utils.rnn import pad_sequence

SEED = 42
DATASET_VERSION = "v3"  # v1 and v2 are superseded; see README "Progress and findings".
RUN_ID = None          # e.g. "phase3-cnn-bigru-v1-0123456789abcdef"; None = newest phase3 run

# Resolve the project root so this notebook runs unchanged on the DGX Spark (container or
# host), a Mac, or any other checkout: env var first, then the Spark paths, then walk up
# from the working directory until a folder containing docs/PLAN.md is found.
def resolve_project_dir():
    candidates = [
        Path(p) for p in (
            os.environ.get("PI_NN_DIR"),
            "/workspace/training/prompt-injection-nn",
            "/home/spark/projects/training/prompt-injection-nn",
        ) if p
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    for parent in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (parent / "docs" / "PLAN.md").exists() and (parent / "notebooks").is_dir():
            return parent
    raise FileNotFoundError(f"project root not found; tried {candidates} and parents of {Path.cwd()}")

def resolve_device():
    # CUDA (DGX Spark) > MPS (Apple Silicon) > CPU. Nothing else in the notebook is device-specific.
    if torch.cuda.is_available():
        return torch.device("cuda")
    mps = getattr(torch.backends, "mps", None)
    if mps is not None and mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def amp_context(device, use_bf16):
    # BF16 autocast only on CUDA. MPS autocast support varies by torch version; CPU gains nothing here.
    if use_bf16 and device.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return nullcontext()


def empty_device_cache(device):
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps" and hasattr(torch, "mps"):
        torch.mps.empty_cache()

# Output contract shared with the LoRA and the Open WebUI filter: "SAFE" or "INJECTION: <reason>".
REASONS = [
    "Override Attempt",
    "Fake System Injection",
    "Meta-Command Injection",
    "Jailbreak Technique",
    "Delimiter Confusion",
    "Privilege Escalation",
    "Embedded Instruction",
    "Role-Play Bypass",
    "Chain-of-Instructions",
]

PROJECT_DIR = resolve_project_dir()
DATA_DIR = PROJECT_DIR / "data" / f"source-clean-{DATASET_VERSION}"
MODELS_DIR = PROJECT_DIR / "models"
if RUN_ID is None:
    candidates = sorted((p for p in MODELS_DIR.glob("phase3-cnn-bigru-*") if (p / "run_manifest.json").exists()),
                        key=lambda p: json.loads((p / "run_manifest.json").read_text())["created_utc"])
    assert candidates, f"no Phase 3 run with a run_manifest.json under {MODELS_DIR}"
    RUN_DIR = candidates[-1]
else:
    RUN_DIR = MODELS_DIR / RUN_ID
run_manifest = json.loads((RUN_DIR / "run_manifest.json").read_text())
config = json.loads((RUN_DIR / "config.json").read_text())
tokenizer_config = json.loads((RUN_DIR / "tokenizer.json").read_text())
thresholds = json.loads((RUN_DIR / "thresholds.json").read_text())
vocab = json.loads((RUN_DIR / "vocab.json").read_text())
TEXT_CONFIG = config["text"]
assert config["reasons"] == REASONS
assert tokenizer_config["vocab_size"] == len(vocab)

data_manifest = json.loads((DATA_DIR / "manifest.json").read_text())
assert data_manifest["config_fingerprint"] == run_manifest["phase1_config_fingerprint"], "the data on disk is not the build this run was trained on"
holdout = pd.read_parquet(DATA_DIR / "lora_holdout.parquet")
expected = data_manifest["counts"]["lora_holdout"]
assert len(holdout) == expected["rows"] and int(holdout["label"].sum()) == expected["injection"]
assert not holdout["augmented"].any()

device = resolve_device()
print({"run": str(RUN_DIR), "fingerprint": run_manifest["phase1_config_fingerprint"][:16], "threshold": thresholds["threshold"],
       "device": str(device), "holdout_rows": len(holdout), "sources": holdout["source"].value_counts().to_dict()})


{'run': '/workspace/training/prompt-injection-nn/models/phase3-cnn-bigru-v2-d3d3fa8d932f3280', 'fingerprint': 'd3d3fa8d932f3280', 'threshold': 0.4330222010612488, 'device': 'cuda', 'holdout_rows': 2086, 'sources': {'wildguard_test': 1286, 'nemotron_test_jailbreaking': 800}}


## 2. Model, Tokenizer and Metric Helpers

Identical to Phase 3. The tokenizer pattern is read back from the run's `tokenizer.json` and asserted equal to the one compiled here.


In [2]:
# Cased, punctuation-preserving tokenizer (PLAN.md section 3). Three token classes: letter runs,
# digit runs, and runs of everything else. "<|im_start|>" becomes "<|", "im", "_", "start", "|>";
# "###" and "---" survive as single tokens; "IGNORE" and "ignore" are different tokens.
TOKEN_PATTERN = re.compile(r"[A-Za-z]+|\d+|[^\sA-Za-z\d]+")
PAD_TOKEN, UNK_TOKEN = "<pad>", "<unk>"


def tokenize(text, max_tokens):
    return TOKEN_PATTERN.findall(text)[:max_tokens]


def encode(text, vocab, max_tokens):
    encoded = [vocab.get(token, vocab[UNK_TOKEN]) for token in tokenize(text, max_tokens)]
    return torch.tensor(encoded or [vocab[UNK_TOKEN]], dtype=torch.long)

class TextCnnBiGru(nn.Module):
    """1D-CNN over token embeddings, bidirectional GRU, masked max-pooling, two heads:
    a binary injection logit and a nine-way reason logit vector."""

    def __init__(self, vocab_size, pad_index, num_reasons, config):
        super().__init__()
        self.pad_index = pad_index
        self.embedding = nn.Embedding(vocab_size, config["embedding_dim"], padding_idx=pad_index)
        self.cnn = nn.Conv1d(config["embedding_dim"], config["cnn_channels"], kernel_size=3, padding=1)
        self.gru = nn.GRU(config["cnn_channels"], config["hidden_dim"], batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(config["dropout"])
        self.shared = nn.Sequential(nn.Linear(config["hidden_dim"] * 2, config["hidden_dim"]), nn.ReLU(), nn.Dropout(config["dropout"]))
        self.injection_head = nn.Linear(config["hidden_dim"], 1)
        self.reason_head = nn.Linear(config["hidden_dim"], num_reasons)

    def forward(self, token_ids):
        mask = token_ids.ne(self.pad_index)
        embedded = self.embedding(token_ids).transpose(1, 2)
        features = torch.relu(self.cnn(embedded)).transpose(1, 2)
        sequence, _ = self.gru(features)
        masked = sequence.masked_fill(~mask.unsqueeze(-1), float("-inf"))
        pooled = self.dropout(masked.max(dim=1).values)
        hidden = self.shared(pooled)
        return self.injection_head(hidden).squeeze(1), self.reason_head(hidden)

def binary_metrics(y_true, probabilities, threshold):
    probabilities = np.asarray(probabilities, dtype=np.float64)
    predictions = (probabilities >= threshold).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    return {
        "threshold": float(threshold),
        "precision": float(precision_score(y_true, predictions, zero_division=0)),
        "recall": float(recall_score(y_true, predictions, zero_division=0)),
        "f1": float(f1_score(y_true, predictions, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, probabilities)) if len(set(y_true)) > 1 else None,
        "average_precision": float(average_precision_score(y_true, probabilities)) if len(set(y_true)) > 1 else None,
        "brier": float(brier_score_loss(y_true, probabilities)),
        "log_loss": float(log_loss(y_true, probabilities, labels=[0, 1])),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "specificity": float(tn / (tn + fp)) if tn + fp else None,
        "false_positive_rate": float(fp / (tn + fp)) if tn + fp else None,
    }


def adversarial_metrics(probabilities, threshold):
    # The adversarial set is all-positive: only recall and the probability distribution are defined.
    probabilities = np.asarray(probabilities, dtype=np.float64)
    return {
        "threshold": float(threshold),
        "injection_recall": float(np.mean(probabilities >= threshold)),
        "probability_mean": float(probabilities.mean()),
        "probability_median": float(np.median(probabilities)),
        "probability_min": float(probabilities.min()),
        "probability_max": float(probabilities.max()),
    }


def select_threshold(y_true, probabilities):
    # Max validation F1; ties broken by higher recall, then lower threshold.
    precision, recall, thresholds = precision_recall_curve(y_true, probabilities)
    candidates = []
    for index, threshold in enumerate(thresholds):
        denominator = precision[index] + recall[index]
        score = 0.0 if denominator == 0 else 2 * precision[index] * recall[index] / denominator
        candidates.append((score, recall[index], -threshold, threshold))
    return float(max(candidates)[3])


FPR_BUDGETS = (0.01, 0.005, 0.001)


def recall_at_fpr(y_true, probabilities, budgets=FPR_BUDGETS):
    # For each FPR budget: the lowest threshold whose FPR on these rows is within budget, and the
    # recall there. This is the deployment table: how much injection is caught per dropped-benign budget.
    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities, dtype=np.float64)
    negatives = np.sort(probabilities[y_true == 0])
    positives = probabilities[y_true == 1]
    table = {}
    for budget in budgets:
        allowed = int(np.floor(budget * len(negatives)))
        # threshold just above the (allowed)-th highest negative score
        threshold = float(np.nextafter(negatives[-allowed - 1], 1.0)) if allowed < len(negatives) else 0.0
        fpr = float(np.mean(negatives >= threshold)) if len(negatives) else 0.0
        table[f"fpr_{budget:g}"] = {
            "threshold": threshold,
            "recall": float(np.mean(positives >= threshold)) if len(positives) else None,
            "achieved_fpr": fpr,
            "allowed_false_positives": allowed,
        }
    return table


def bootstrap_ci(y_true, probabilities, threshold, n_resamples=1000, seed=42, alpha=0.05):
    # Percentile bootstrap over rows. Reported with every locked-test number so a single point
    # estimate is never presented as the result.
    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities, dtype=np.float64)
    rng = np.random.default_rng(seed)
    predictions = (probabilities >= threshold).astype(np.int64)
    samples = {"average_precision": [], "roc_auc": [], "f1": [], "recall": [], "false_positive_rate": []}
    n = len(y_true)
    for _ in range(n_resamples):
        index = rng.integers(0, n, n)
        yt, pr, pd_ = y_true[index], probabilities[index], predictions[index]
        if yt.min() == yt.max():
            continue
        tp = int(((pd_ == 1) & (yt == 1)).sum()); fp = int(((pd_ == 1) & (yt == 0)).sum())
        fn = int(((pd_ == 0) & (yt == 1)).sum()); tn = int(((pd_ == 0) & (yt == 0)).sum())
        samples["average_precision"].append(average_precision_score(yt, pr))
        samples["roc_auc"].append(roc_auc_score(yt, pr))
        samples["f1"].append(2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0)
        samples["recall"].append(tp / (tp + fn) if tp + fn else 0.0)
        samples["false_positive_rate"].append(fp / (fp + tn) if fp + tn else 0.0)
    lo, hi = 100 * alpha / 2, 100 * (1 - alpha / 2)
    return {
        key: {"low": float(np.percentile(values, lo)), "high": float(np.percentile(values, hi)), "resamples": len(values)}
        for key, values in samples.items()
    }


def slice_metrics(frame, probabilities, threshold, column):
    # Per-slice metrics. Single-class slices (all SAFE or all INJECTION) report only what is defined.
    rows = {}
    probabilities = np.asarray(probabilities, dtype=np.float64)
    for value, index in frame.groupby(column).indices.items():
        y = frame["label"].to_numpy()[index]
        p = probabilities[index]
        predictions = (p >= threshold).astype(int)
        entry = {"rows": int(len(index)), "injection_rows": int(y.sum()), "safe_rows": int((y == 0).sum())}
        if y.sum():
            entry["recall"] = float(predictions[y == 1].mean())
        if (y == 0).sum():
            entry["false_positive_rate"] = float(predictions[y == 0].mean())
        if 0 < y.sum() < len(y):
            entry["f1"] = float(f1_score(y, predictions, zero_division=0))
            entry["average_precision"] = float(average_precision_score(y, p))
        rows[str(value)] = entry
    return rows


def atomic_json(payload, path):
    temporary = Path(str(path) + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str))
    os.replace(temporary, path)


assert tokenizer_config["token_pattern"] == TOKEN_PATTERN.pattern, "tokenizer drift between this notebook and the run"
PAD_INDEX = tokenizer_config["pad_index"]
model = TextCnnBiGru(len(vocab), PAD_INDEX, len(REASONS), TEXT_CONFIG)
model.load_state_dict(torch.load(RUN_DIR / "model_state.pt", map_location="cpu"))
model.to(device).eval()
use_bf16 = device.type == "cuda" and torch.cuda.is_bf16_supported()


def predict_texts(text_values, batch_size=256):
    probabilities, reason_logits = [], []
    with torch.inference_mode():
        for start in range(0, len(text_values), batch_size):
            batch = [encode(t, vocab, TEXT_CONFIG["max_tokens"]) for t in text_values[start:start + batch_size]]
            token_ids = pad_sequence(batch, batch_first=True, padding_value=PAD_INDEX).to(device)
            with amp_context(device, use_bf16):
                injection_logit, reason_logit = model(token_ids)
            probabilities.append(torch.sigmoid(injection_logit.float()).cpu().numpy())
            reason_logits.append(reason_logit.float().cpu().numpy())
    return np.concatenate(probabilities), np.concatenate(reason_logits)


def contract_string(probability, reason_logit, threshold):
    # The output contract shared with the LoRA and the filter. A reason is reported only when the
    # injection head fires; it is always one of the nine, so "invalid" can only ever be zero here.
    if probability < threshold:
        return "SAFE"
    return f"INJECTION: {REASONS[int(np.argmax(reason_logit))]}"


def parse_pi(raw):
    # Copied from the LoRA eval notebook so both sides parse their own output the same way.
    first = raw.splitlines()[0].strip() if raw else ""
    if first == "SAFE":
        return "SAFE", None, True
    if first.startswith("INJECTION:"):
        reason = first[len("INJECTION:"):].strip()
        return "INJECTION", reason, reason in REASONS
    return None, None, False


## 3. Score the Held-Out Rows and Write the Eval Files


In [3]:
threshold = float(thresholds["threshold"])
texts = holdout["text"].fillna("").astype(str).tolist()
probabilities, reason_logits = predict_texts(texts)
y_true = holdout["label"].to_numpy(dtype=np.int64)

eval_rows = []
for i, row in enumerate(holdout.itertuples(index=False)):
    raw = contract_string(probabilities[i], reason_logits[i], threshold)
    pred, reason, valid = parse_pi(raw)
    eval_rows.append({"prompt": row.text, "gold": "INJECTION" if row.label == 1 else "SAFE", "gold_reason": row.reason or None,
                      "source": row.source, "pred": pred, "reason": reason, "valid": valid, "raw": raw,
                      "probability": float(probabilities[i])})

n = len(eval_rows)
invalid = sum(1 for r in eval_rows if not r["valid"])
acc = sum(1 for r in eval_rows if r["pred"] == r["gold"]) / n
safe_rows = [r for r in eval_rows if r["gold"] == "SAFE"]
inj_rows = [r for r in eval_rows if r["gold"] == "INJECTION"]
fp = sum(1 for r in safe_rows if r["pred"] == "INJECTION")
fn = sum(1 for r in inj_rows if r["pred"] == "SAFE")

print(f"==== PROMPT INJECTION EVAL ({n} rows) ====")
print(f"  Accuracy (SAFE/INJECTION):            {acc:.3f}")
print(f"  Invalid output:                        {invalid} ({invalid / n:.1%})")
print(f"  False positives (SAFE -> INJECTION):  {fp} / {len(safe_rows)} = {fp / max(1, len(safe_rows)):.1%}")
print(f"  False negatives (INJECTION -> SAFE):  {fn} / {len(inj_rows)} = {fn / max(1, len(inj_rows)):.1%}")
print("  By source:")
by_source = {}
for src in sorted(set(r["source"] for r in eval_rows)):
    grp = [r for r in eval_rows if r["source"] == src]
    ok = sum(1 for r in grp if r["pred"] == r["gold"])
    by_source[src] = {"rows": len(grp), "correct": ok, "accuracy": ok / len(grp)}
    print(f"    {src:<32} {ok}/{len(grp)} = {ok / len(grp):.1%}")
print(f"  Predicted reasons: {Counter(r['reason'] for r in eval_rows if r['reason'])}")

# Probability-based metrics the LoRA cannot produce, reported alongside the shared fields.
probability_metrics = {
    "binary": binary_metrics(y_true, probabilities, threshold),
    "bootstrap_95ci": bootstrap_ci(y_true, probabilities, threshold, seed=SEED),
    "recall_at_fpr": recall_at_fpr(y_true, probabilities),
    "slices_by_source": slice_metrics(holdout, probabilities, threshold, "source"),
}
print("  95% CI:", {k: (round(v["low"], 4), round(v["high"], 4)) for k, v in probability_metrics["bootstrap_95ci"].items()})
print("  recall at FPR budget:", {k: round(v["recall"], 4) for k, v in probability_metrics["recall_at_fpr"].items()})

results = {
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "adapter": None,
    "base_model": "none (CNN+BiGRU, no LLM)",
    "model_run": str(RUN_DIR), "run_id": run_manifest["run_id"],
    "dataset_fingerprint": run_manifest["phase1_config_fingerprint"], "threshold": threshold,
    "rows": n, "accuracy": acc, "invalid": invalid,
    "false_positives": fp, "safe_rows": len(safe_rows), "false_negatives": fn, "injection_rows": len(inj_rows),
    "sources": dict(Counter(r["source"] for r in eval_rows)),
    "accuracy_by_source": by_source,
    "probability_metrics": probability_metrics,
    "device": str(device), "torch": package_version("torch"),
}
out_path = RUN_DIR / "eval_wildguard_test.json"
atomic_json(results, out_path)
(RUN_DIR / "eval_wildguard_test_rows.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in eval_rows) + "\n", encoding="utf-8")
print(f"\nSaved: {out_path}  (+ per-row file beside it)")


==== PROMPT INJECTION EVAL (2086 rows) ====
  Accuracy (SAFE/INJECTION):            0.901
  Invalid output:                        0 (0.0%)
  False positives (SAFE -> INJECTION):  89 / 945 = 9.4%
  False negatives (INJECTION -> SAFE):  118 / 1141 = 10.3%
  By source:
    nemotron_test_jailbreaking       776/800 = 97.0%
    wildguard_test                   1103/1286 = 85.8%
  Predicted reasons: Counter({'Jailbreak Technique': 1055, 'Privilege Escalation': 50, 'Override Attempt': 4, 'Meta-Command Injection': 2, 'Embedded Instruction': 1})
  95% CI: {'average_precision': (0.9605, 0.9766), 'roc_auc': (0.9569, 0.9715), 'f1': (0.8968, 0.9199), 'recall': (0.8793, 0.9135), 'false_positive_rate': (0.0762, 0.1129)}
  recall at FPR budget: {'fpr_0.01': 0.5259, 'fpr_0.005': 0.3742, 'fpr_0.001': 0.2226}

Saved: /workspace/training/prompt-injection-nn/models/phase3-cnn-bigru-v2-d3d3fa8d932f3280/eval_wildguard_test.json  (+ per-row file beside it)


## 4. Report

Writes `reports/<UTC stamp>_phase_4_eval_lora_holdouts_<version>_<fingerprint>/report.md`: the LoRA-format block
(accuracy, invalid, false positives, false negatives, by source), the probability-based metrics
with bootstrap intervals and recall at the FPR budgets, and the predicted-reason distribution.


In [4]:
# ---- Report: tracked Markdown summary of this run, written to reports/ with its plots beside it ----
import shutil

REPORTS_DIR = PROJECT_DIR / "reports"
RUN_STAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%SZ")


def report_dir(run_name):
    # One folder per run: reports/<UTC stamp>_<phase>_<version>_<fingerprint>/ holding report.md and its plots.
    path = REPORTS_DIR / f"{RUN_STAMP}_{run_name}"
    path.mkdir(parents=True, exist_ok=True)
    return path


def md_table(frame, floatfmt="{:.4f}", index=True):
    frame = pd.DataFrame(frame)
    cols = [str(c) for c in frame.columns]
    header = ([""] if index else []) + cols
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    for idx, row in frame.iterrows():
        cells = []
        for v in row.tolist():
            if v is None or (isinstance(v, float) and np.isnan(v)):
                cells.append("")
            elif isinstance(v, (float, np.floating)):
                cells.append(floatfmt.format(float(v)))
            elif isinstance(v, (bool, np.bool_)):
                cells.append("yes" if v else "no")
            else:
                cells.append(str(v).replace("|", "\\|").replace("\n", " "))
        lines.append("| " + " | ".join(([str(idx)] if index else []) + cells) + " |")
    return "\n".join(lines)


def write_report(name, title, sections):
    # sections: list of (heading, markdown body). Atomic write, LF endings.
    parts = [f"# {title}", f"_Generated {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}_"]
    for heading, body in sections:
        parts.append(f"## {heading}\n\n{body}" if heading else body)
    path = report_dir(name) / "report.md"
    tmp = Path(str(path) + ".tmp")
    tmp.write_text("\n\n".join(parts) + "\n", encoding="utf-8", newline="\n")
    os.replace(tmp, path)
    print("report written:", path)
    return path


def report_asset(run_name, filename):
    # Plots live beside report.md and are referenced by bare file name.
    return report_dir(run_name), filename

run_name = f"phase_4_eval_lora_holdouts_{DATASET_VERSION}_{run_manifest['phase1_config_fingerprint'][:16]}"
headline_cols = ["threshold", "precision", "recall", "f1", "roc_auc", "average_precision", "brier", "log_loss", "false_positive_rate"]
ci = {k: f"{v['low']:.4f} to {v['high']:.4f}" for k, v in probability_metrics["bootstrap_95ci"].items()}
rafpr = {budget: {"recall": row["recall"], "threshold": row["threshold"], "achieved_fpr": row["achieved_fpr"]} for budget, row in probability_metrics["recall_at_fpr"].items()}
source_rows = {source: {m: row.get(m) for m in ("rows", "recall", "false_positive_rate", "f1")} for source, row in probability_metrics["slices_by_source"].items()}
predicted_reasons = pd.Series(Counter(r["reason"] for r in eval_rows if r["reason"]), name="predicted").to_frame()

write_report(run_name, f"Phase 4 Report: Evaluation on the LoRA held-out sets ({run_manifest['run_id']})", [
    ("Run", f"- Phase 3 run: `{RUN_DIR.relative_to(PROJECT_DIR)}`\n- Dataset fingerprint: `{run_manifest['phase1_config_fingerprint']}`\n- Threshold: {threshold:.4f}\n- Held-out rows: {n:,} ({', '.join(f'{k} {v:,}' for k, v in results['sources'].items())})\n- Output: `eval_wildguard_test.json` and `eval_wildguard_test_rows.jsonl` in the run directory"),
    ("LoRA-format summary", f"- Accuracy (SAFE/INJECTION): {acc:.3f}\n- Invalid output: {invalid} ({invalid / n:.1%})\n- False positives (SAFE -> INJECTION): {fp} / {len(safe_rows)} = {fp / max(1, len(safe_rows)):.1%}\n- False negatives (INJECTION -> SAFE): {fn} / {len(inj_rows)} = {fn / max(1, len(inj_rows)):.1%}\n\n" + md_table(pd.DataFrame(by_source).T, floatfmt="{:.3f}")),
    ("Probability-based metrics", md_table(pd.DataFrame({"lora_holdout": probability_metrics["binary"]}).T[headline_cols]) + "\n\n95% bootstrap intervals (1,000 resamples):\n\n" + md_table(pd.DataFrame({"lora_holdout": ci}).T)),
    ("Recall at FPR budget", md_table(pd.DataFrame(rafpr).T)),
    ("Slices by source", md_table(pd.DataFrame(source_rows).T)),
    ("Predicted reasons on flagged rows", md_table(predicted_reasons, floatfmt="{:.0f}")),
    ("Comparison status", "No LoRA `eval_*.json` exists yet in the `safety` repo; this report is one side of the head-to-head. The LoRA eval notebook writes the same `rows`, `accuracy`, `invalid`, `false_positives`, `safe_rows`, `false_negatives`, `injection_rows` and `sources` fields."),
])


report written: /workspace/training/prompt-injection-nn/reports/2026-10-09_170917Z_phase_4_eval_lora_holdouts_v2_d3d3fa8d932f3280/report.md


PosixPath('/workspace/training/prompt-injection-nn/reports/2026-10-09_170917Z_phase_4_eval_lora_holdouts_v2_d3d3fa8d932f3280/report.md')